MIN-HASHING AND LSH

In [3]:
import os
import time
import math
import hashlib
from itertools import combinations
from collections import defaultdict

import numpy as np
import pandas as pd

1. Create k-Grams [20]

In [11]:

def exact_jaccard(set_a, set_b):
    """Compute exact Jaccard similarity between two sets."""
    if len(set_a) == 0 and len(set_b) == 0:
        return 1.0
    return len(set_a & set_b) / len(set_a | set_b)



def stable_hash_str(x, mod_val):
    """
    Stable hash using SHA1 (Python built-in hash changes across sessions).
    """
    h = hashlib.sha1(str(x).encode("utf-8")).hexdigest()
    return int(h[:16], 16) % mod_val



def next_prime(n):
    def is_prime(v):
        if v < 2:
            return False
        if v % 2 == 0:
            return v == 2
        r = int(v ** 0.5)
        for i in range(3, r + 1, 2):
            if v % i == 0:
                return False
        return True

    p = max(2, n + 1)
    while not is_prime(p):
        p += 1
    return p

In [12]:
def load_docs(minhash_dir):
    """
    Load D1.txt, D2.txt, D3.txt, D4.txt from minhash directory.
    Returns dict like {"D1": "...", "D2": "..."}.
    """
    docs = {}
    for fname in ["D1.txt", "D2.txt", "D3.txt", "D4.txt"]:
        fpath = os.path.join(minhash_dir, fname)
        with open(fpath, "r", encoding="utf-8") as f:
            docs[fname.replace(".txt", "")] = f.read().rstrip("\n")
    return docs

In [13]:
def char_kgrams(text, k):
    """
    Character k-grams. Space is counted as a character automatically.
    Duplicate k-grams removed using set().
    """
    if len(text) < k:
        return set()
    return set(text[i:i+k] for i in range(len(text) - k + 1))

In [14]:
def char_kgrams(text, k):
    """
    Character k-grams. Space is counted as a character automatically.
    Duplicate k-grams removed using set().
    """
    if len(text) < k:
        return set()
    return set(text[i:i+k] for i in range(len(text) - k + 1))

In [15]:
def build_all_kgram_sets(docs):
    """
    Build:
      - char_2
      - char_3
      - word_2
    for all documents.
    """
    gram_sets = {
        "char_2": {},
        "char_3": {},
        "word_2": {}
    }

    for dname, text in docs.items():
        gram_sets["char_2"][dname] = char_kgrams(text, 2)
        gram_sets["char_3"][dname] = char_kgrams(text, 3)
        gram_sets["word_2"][dname] = word_kgrams(text, 2)

    return gram_sets

In [16]:
def build_all_kgram_sets(docs):
    """
    Build:
      - char_2
      - char_3
      - word_2
    for all documents.
    """
    gram_sets = {
        "char_2": {},
        "char_3": {},
        "word_2": {}
    }

    for dname, text in docs.items():
        gram_sets["char_2"][dname] = char_kgrams(text, 2)
        gram_sets["char_3"][dname] = char_kgrams(text, 3)
        gram_sets["word_2"][dname] = word_kgrams(text, 2)

    return gram_sets

In [18]:
def compute_pairwise_jaccards(gram_sets):
    """
    Returns dict:
    {
      "char_2": {("D1","D2"): value, ...},
      "char_3": {...},
      "word_2": {...}
    }
    """
    results = {}
    doc_names = ["D1", "D2", "D3", "D4"]

    for gtype, gmap in gram_sets.items():
        pair_sims = {}
        for a, b in combinations(doc_names, 2):
            pair_sims[(a, b)] = exact_jaccard(gmap[a], gmap[b])
        results[gtype] = pair_sims

    return results

In [19]:

minhash_dir = r"C:\Nitin Awasthi\M-Tech\Semester-5\Big Data\CSL7110_Assignment\minhash"   

docs = load_docs(minhash_dir)
gram_sets = build_all_kgram_sets(docs)
jaccard_results = compute_pairwise_jaccards(gram_sets)


print("Document lengths:")
for d in ["D1", "D2", "D3", "D4"]:
    print(d, "->", len(docs[d]))


print("\nDistinct k-gram counts (duplicates removed):")
for gtype in ["char_2", "char_3", "word_2"]:
    print("\n", gtype)
    for d in ["D1", "D2", "D3", "D4"]:
        print(" ", d, ":", len(gram_sets[gtype][d]))


print("\n=== Point 1 (18 Jaccard similarities) ===")
for gtype in ["char_2", "char_3", "word_2"]:
    print("\n", gtype)
    for pair, sim in jaccard_results[gtype].items():
        print(" ", pair, "=", round(sim, 6))

Document lengths:
D1 -> 1749
D2 -> 1747
D3 -> 2132
D4 -> 1435

Distinct k-gram counts (duplicates removed):

 char_2
  D1 : 263
  D2 : 262
  D3 : 269
  D4 : 255

 char_3
  D1 : 765
  D2 : 762
  D3 : 828
  D4 : 698

 word_2
  D1 : 279
  D2 : 278
  D3 : 337
  D4 : 232

=== Point 1 (18 Jaccard similarities) ===

 char_2
  ('D1', 'D2') = 0.981132
  ('D1', 'D3') = 0.8157
  ('D1', 'D4') = 0.644444
  ('D2', 'D3') = 0.8
  ('D2', 'D4') = 0.64127
  ('D3', 'D4') = 0.652997

 char_3
  ('D1', 'D2') = 0.977979
  ('D1', 'D3') = 0.580357
  ('D1', 'D4') = 0.305085
  ('D2', 'D3') = 0.568047
  ('D2', 'D4') = 0.305903
  ('D3', 'D4') = 0.312124

 word_2
  ('D1', 'D2') = 0.940767
  ('D1', 'D3') = 0.182342
  ('D1', 'D4') = 0.030242
  ('D2', 'D3') = 0.173664
  ('D2', 'D4') = 0.030303
  ('D3', 'D4') = 0.016071


In [22]:
rows = []
for pair in combinations(["D1", "D2", "D3", "D4"], 2):
    rows.append({
        "Pair": f"{pair[0]}-{pair[1]}",
        "Jaccard_char_2": jaccard_results["char_2"][pair],
        "Jaccard_char_3": jaccard_results["char_3"][pair],
        "Jaccard_word_2": jaccard_results["word_2"][pair],
    })

df_point1 = pd.DataFrame(rows)
print(df_point1)

# Save
os.makedirs(r"C:\Nitin Awasthi\M-Tech\Semester-5\Big Data\CSL7110_Assignment\outputs", exist_ok=True)
df_point1.to_csv(r"C:\Nitin Awasthi\M-Tech\Semester-5\Big Data\CSL7110_Assignment\outputs\point1_jaccard_18_values.csv", index=False)
print("Saved: outputs/point1_jaccard_18_values.csv")

    Pair  Jaccard_char_2  Jaccard_char_3  Jaccard_word_2
0  D1-D2        0.981132        0.977979        0.940767
1  D1-D3        0.815700        0.580357        0.182342
2  D1-D4        0.644444        0.305085        0.030242
3  D2-D3        0.800000        0.568047        0.173664
4  D2-D4        0.641270        0.305903        0.030303
5  D3-D4        0.652997        0.312124        0.016071
Saved: outputs/point1_jaccard_18_values.csv


2.Min-Hashing [20]

Build MinHash hash family

In [34]:
def build_hash_family(t, m=50021, seed=42):
    """
    Build t hash functions of the form:
      h(x) = ((a*x + b) mod p) mod m
    where m > 10000 (assignment condition satisfied)
    """
    rng = np.random.default_rng(seed)
    p = next_prime(m * 10)   # prime > m
    a = rng.integers(1, p, size=t, dtype=np.int64)
    b = rng.integers(0, p, size=t, dtype=np.int64)
    return a, b, p, m

Convert shingles (3-grams) into integer universe

In [35]:
def shingles_to_ints(shingle_set, m=50021):
    """
    Convert k-grams (strings/tuples) to stable integer IDs using SHA1 mod m.
    """
    return set(stable_hash_str(s, m) for s in shingle_set)

Build MinHash signature for one set

In [36]:
def minhash_signature_from_intset(int_set, t, seed=42, m=50021):
    """
    Compute MinHash signature of length t for a set of integers.
    """
    a, b, p, m = build_hash_family(t, m=m, seed=seed)

    if len(int_set) == 0:
        return np.full(t, np.iinfo(np.int64).max, dtype=np.int64)

    arr = np.array(list(int_set), dtype=np.int64)  # elements in set


    hashed = ((a[:, None] * arr[None, :] + b[:, None]) % p) % m


    signature = hashed.min(axis=1)
    return signature

Estimate Jaccard from two signatures

In [37]:
def minhash_estimate(sig1, sig2):
    """
    Approx Jaccard = fraction of positions where signatures match.
    """
    return float(np.mean(sig1 == sig2))

Run Point 2 for D1 and D2 using char 3-grams

In [38]:

d1_3 = gram_sets["char_3"]["D1"]
d2_3 = gram_sets["char_3"]["D2"]

# Exact Jaccard (ground truth)
exact_d1d2_char3 = exact_jaccard(d1_3, d2_3)
print("Exact Jaccard (D1,D2) using char 3-grams =", round(exact_d1d2_char3, 6))

# Convert to integer sets
d1_3_ints = shingles_to_ints(d1_3, m=50021)
d2_3_ints = shingles_to_ints(d2_3, m=50021)

t_values = [20, 60, 150, 300, 600]

rows = []
for t in t_values:
    start = time.perf_counter()

    sig1 = minhash_signature_from_intset(d1_3_ints, t=t, seed=123, m=50021)
    sig2 = minhash_signature_from_intset(d2_3_ints, t=t, seed=123, m=50021)

    approx = minhash_estimate(sig1, sig2)
    elapsed = time.perf_counter() - start

    rows.append({
        "t": t,
        "Approx_Jaccard": approx,
        "Exact_Jaccard": exact_d1d2_char3,
        "Abs_Error": abs(approx - exact_d1d2_char3),
        "Time_sec": elapsed
    })

df_point2 = pd.DataFrame(rows)
print(df_point2)

df_point2.to_csv(r"C:\Nitin Awasthi\M-Tech\Semester-5\Big Data\CSL7110_Assignment\outputs\point2_minhash_d1_d2_char3.csv", index=False)
print("Saved: outputs/point2_minhash_d1_d2_char3.csv")

Exact Jaccard (D1,D2) using char 3-grams = 0.977979
     t  Approx_Jaccard  Exact_Jaccard  Abs_Error  Time_sec
0   20        0.950000       0.977979   0.027979  0.005060
1   60        0.983333       0.977979   0.005354  0.004246
2  150        0.973333       0.977979   0.004646  0.008868
3  300        0.976667       0.977979   0.001313  0.022108
4  600        0.988333       0.977979   0.010354  0.049515
Saved: outputs/point2_minhash_d1_d2_char3.csv


(Optional but good) Run multiple times to justify best t

In [39]:

analysis_rows = []

for t in t_values:
    errs = []
    times = []

    for run_seed in [100, 101, 102, 103, 104]:
        start = time.perf_counter()

        sig1 = minhash_signature_from_intset(d1_3_ints, t=t, seed=run_seed, m=50021)
        sig2 = minhash_signature_from_intset(d2_3_ints, t=t, seed=run_seed, m=50021)

        approx = minhash_estimate(sig1, sig2)
        elapsed = time.perf_counter() - start

        errs.append(abs(approx - exact_d1d2_char3))
        times.append(elapsed)

    analysis_rows.append({
        "t": t,
        "Mean_Abs_Error_5runs": float(np.mean(errs)),
        "Std_Abs_Error_5runs": float(np.std(errs)),
        "Mean_Time_sec_5runs": float(np.mean(times)),
    })

df_point2b = pd.DataFrame(analysis_rows)
print(df_point2b)
df_point2b.to_csv(r"C:\Nitin Awasthi\M-Tech\Semester-5\Big Data\CSL7110_Assignment\outputs\point2b_t_selection_analysis.csv", index=False)
print("Saved: outputs/point2b_t_selection_analysis.csv")

     t  Mean_Abs_Error_5runs  Std_Abs_Error_5runs  Mean_Time_sec_5runs
0   20              0.022021             0.000000             0.001187
1   60              0.008687             0.006667             0.003695
2  150              0.010546             0.005621             0.006331
3  300              0.009454             0.005704             0.025183
4  600              0.001596             0.000871             0.039237
Saved: outputs/point2b_t_selection_analysis.csv


3: LSH [20]

LSH probability function

In [42]:
def lsh_candidate_probability(s, b_rows_per_band, r_num_bands):
    """
    Assignment formula:
        f(s) = 1 - (1 - s^b)^r
    where:
      b = rows per band
      r = number of bands
    """
    return 1.0 - (1.0 - (s ** b_rows_per_band)) ** r_num_bands

Test all valid (b, r) pairs for t = 160

In [43]:
def lsh_param_search_t160(t=160, tau=0.7):
    rows = []

    for b_rows_per_band in range(1, t + 1):
        if t % b_rows_per_band != 0:
            continue

        r_num_bands = t // b_rows_per_band
        p_at_tau = lsh_candidate_probability(tau, b_rows_per_band, r_num_bands)

        # approximate "knee" threshold (rough)
        approx_knee = (1 / r_num_bands) ** (1 / b_rows_per_band)

        rows.append({
            "b_rows_per_band": b_rows_per_band,
            "r_num_bands": r_num_bands,
            "b_times_r": b_rows_per_band * r_num_bands,
            "P_at_tau_0.7": p_at_tau,
            "Approx_knee": approx_knee,
            "Dist_knee_from_tau": abs(approx_knee - tau)
        })

    df = pd.DataFrame(rows).sort_values("Dist_knee_from_tau")
    return df


df_lsh_params = lsh_param_search_t160(t=160, tau=0.7)
print(df_lsh_params.head(12))  # top candidates
df_lsh_params.to_csv(r"C:\Nitin Awasthi\M-Tech\Semester-5\Big Data\CSL7110_Assignment\outputs\point3_lsh_param_search_t160.csv", index=False)
print("Saved: outputs/point3_lsh_param_search_t160.csv")

    b_rows_per_band  r_num_bands  b_times_r  P_at_tau_0.7  Approx_knee  \
4                 8           20        160  6.950258e-01     0.687656   
5                10           16        160  3.677476e-01     0.757858   
6                16           10        160  3.274032e-02     0.865964   
3                 5           32        160  9.972281e-01     0.500000   
7                20            8        160  6.365583e-03     0.901250   
8                32            5        160  5.522016e-05     0.950949   
9                40            4        160  2.546720e-06     0.965936   
10               80            2        160  8.106849e-13     0.991373   
11              160            1        160  0.000000e+00     1.000000   
2                 4           40        160  9.999830e-01     0.397635   
1                 2           80        160  1.000000e+00     0.111803   
0                 1          160        160  1.000000e+00     0.006250   

    Dist_knee_from_tau  
4           

Choose final (b, r) and compute probability for all 6 doc pairs

In [44]:

b_rows_per_band = 8
r_num_bands = 20

rows = []
for pair, s in jaccard_results["char_3"].items():   # 6 pairs
    p_candidate = lsh_candidate_probability(s, b_rows_per_band, r_num_bands)
    rows.append({
        "Pair": f"{pair[0]}-{pair[1]}",
        "Exact_Jaccard_char3": s,
        "LSH_candidate_probability": p_candidate
    })

df_point3 = pd.DataFrame(rows)
print(df_point3)

df_point3.to_csv(r"C:\Nitin Awasthi\M-Tech\Semester-5\Big Data\CSL7110_Assignment\outputs\point3_pair_probabilities.csv", index=False)
print("Saved: outputs/point3_pair_probabilities.csv")

    Pair  Exact_Jaccard_char3  LSH_candidate_probability
0  D1-D2             0.977979                   1.000000
1  D1-D3             0.580357                   0.228224
2  D1-D4             0.305085                   0.001500
3  D2-D3             0.568047                   0.195880
4  D2-D4             0.305903                   0.001532
5  D3-D4             0.312124                   0.001800
Saved: outputs/point3_pair_probabilities.csv


4. Min-Hashing on MovieLens dataset [20]

Load u.data

In [46]:
def load_movielens_u_data(u_data_path):
    """
    Read MovieLens 100k u.data file:
    user_id, movie_id, rating, timestamp (tab-separated)
    """
    df = pd.read_csv(
        u_data_path,
        sep="\t",
        header=None,
        names=["user", "movie", "rating", "timestamp"]
    )

    # Keep only movie set per user
    user_movies = df.groupby("user")["movie"].apply(lambda x: set(map(int, x.tolist()))).to_dict()
    users = sorted(user_movies.keys())

    return df, user_movies, users

Compute exact Jaccard for all user pairs

In [47]:
def exact_all_user_pairs(user_movies, users):
    """
    Returns dict {(u1,u2): exact_jaccard}
    """
    exact = {}
    for u1, u2 in combinations(users, 2):
        exact[(u1, u2)] = exact_jaccard(user_movies[u1], user_movies[u2])
    return exact

MinHash optimization for MovieLens (precompute movie hash table)

In [49]:
def build_minhash_value_table_for_movies(max_movie_id, t, seed=42, m=50021):
    """
    Precompute hashed values for movie IDs for all t hash functions.
    shape = (t, max_movie_id+1)
    """
    a, b, p, m = build_hash_family(t, m=m, seed=seed)

    movie_ids = np.arange(max_movie_id + 1, dtype=np.int64)
    values = ((a[:, None] * movie_ids[None, :] + b[:, None]) % p) % m
    return values

Build user signatures from precomputed table

In [50]:
def user_signatures_from_movie_value_table(user_movies, users, value_table):
    """
    Build MinHash signature for each user using precomputed movie hash values.
    """
    t = value_table.shape[0]
    sigs = {}

    for u in users:
        movie_ids = np.array(sorted(list(user_movies[u])), dtype=np.int64)
        if len(movie_ids) == 0:
            sigs[u] = np.full(t, np.iinfo(np.int64).max, dtype=np.int64)
        else:
            sigs[u] = value_table[:, movie_ids].min(axis=1)

    return sigs

Approx Jaccard from signatures for all user pairs

In [52]:
def approx_all_user_pairs_from_sigs(sigs, users):
    """
    Returns dict {(u1,u2): approx_jaccard}
    """
    approx = {}
    for u1, u2 in combinations(users, 2):
        approx[(u1, u2)] = float(np.mean(sigs[u1] == sigs[u2]))
    return approx

FP/FN helper

In [53]:
def fp_fn_counts_from_similarity_dicts(exact_dict, approx_dict, threshold):
    """
    exact_dict: {(u1,u2): exact_similarity}
    approx_dict: {(u1,u2): estimated_similarity}
    """
    exact_pos = {pair for pair, s in exact_dict.items() if s >= threshold}
    pred_pos = {pair for pair, s in approx_dict.items() if s >= threshold}

    fp = len(pred_pos - exact_pos)
    fn = len(exact_pos - pred_pos)
    return fp, fn, exact_pos, pred_pos

Run Point 4 end-to-end

In [57]:
# ---- CHANGE THIS PATH ----
u_data_path = r"C:\Nitin Awasthi\M-Tech\Semester-5\Big Data\CSL7110_Assignment\ml-100k\ml-100k\u.data"   # Example path

df_ml, user_movies, users = load_movielens_u_data(u_data_path)
print("Users:", len(users))
print("Ratings rows:", len(df_ml))
print("Max movie ID:", df_ml["movie"].max())

# 1) Exact Jaccard for all user pairs
start = time.perf_counter()
exact_ml = exact_all_user_pairs(user_movies, users)
elapsed_exact = time.perf_counter() - start
print("Exact all-pairs Jaccard done in", round(elapsed_exact, 2), "sec")

# Save exact pairs >= 0.5
exact_pairs_ge_05 = [(u1, u2, s) for (u1, u2), s in exact_ml.items() if s >= 0.5]
exact_pairs_ge_05.sort(key=lambda x: (-x[2], x[0], x[1]))

df_exact_05 = pd.DataFrame(exact_pairs_ge_05, columns=["user1", "user2", "exact_jaccard"])
df_exact_05.to_csv(r"C:\Nitin Awasthi\M-Tech\Semester-5\Big Data\CSL7110_Assignment\outputs\point4_exact_pairs_ge_0_5.csv", index=False)
print("Exact pairs >=0.5:", len(df_exact_05))
print("Saved: outputs/point4_exact_pairs_ge_0_5.csv")


# 2) MinHash for t = 50, 100, 200 (5 runs average FP/FN)
t_values = [50, 100, 200]
summary_rows = []

max_movie_id = int(df_ml["movie"].max())

for t in t_values:
    fp_runs = []
    fn_runs = []

    print("\nRunning MinHash for t =", t)

    for run_idx, seed in enumerate([101, 102, 103, 104, 105], start=1):
        value_table = build_minhash_value_table_for_movies(max_movie_id, t=t, seed=seed, m=50021)
        sigs = user_signatures_from_movie_value_table(user_movies, users, value_table)

        approx_ml = approx_all_user_pairs_from_sigs(sigs, users)

        fp, fn, exact_pos, pred_pos = fp_fn_counts_from_similarity_dicts(exact_ml, approx_ml, threshold=0.5)
        fp_runs.append(fp)
        fn_runs.append(fn)

        # Save one run's predicted pairs for reporting (last run)
        if run_idx == 5:
            pred_pairs = [(u1, u2, s) for (u1, u2), s in approx_ml.items() if s >= 0.5]
            pred_pairs.sort(key=lambda x: (-x[2], x[0], x[1]))
            df_pred = pd.DataFrame(pred_pairs, columns=["user1", "user2", "estimated_jaccard"])
            out_file = r"C:\Nitin Awasthi\M-Tech\Semester-5\Big Data\CSL7110_Assignment\outputs\point4_minhash_pairs_ge_0_5_t{t}.csv"
            df_pred.to_csv(out_file, index=False)
            print("Saved:", out_file)

        print(f"  Run {run_idx}: FP={fp}, FN={fn}")

    summary_rows.append({
        "t": t,
        "Avg_FP_5runs": float(np.mean(fp_runs)),
        "Avg_FN_5runs": float(np.mean(fn_runs)),
        "FP_runs": str(fp_runs),
        "FN_runs": str(fn_runs)
    })

df_point4_summary = pd.DataFrame(summary_rows)
print("\nPoint 4 Summary:")
print(df_point4_summary)

df_point4_summary.to_csv(r"C:\Nitin Awasthi\M-Tech\Semester-5\Big Data\CSL7110_Assignment\outputs\point4_minhash_fp_fn_summary.csv", index=False)
print("Saved: outputs/point4_minhash_fp_fn_summary.csv")

Users: 943
Ratings rows: 100000
Max movie ID: 1682
Exact all-pairs Jaccard done in 11.13 sec
Exact pairs >=0.5: 10
Saved: outputs/point4_exact_pairs_ge_0_5.csv

Running MinHash for t = 50
  Run 1: FP=69, FN=0
  Run 2: FP=237, FN=2
  Run 3: FP=88, FN=1
  Run 4: FP=117, FN=4
Saved: C:\Nitin Awasthi\M-Tech\Semester-5\Big Data\CSL7110_Assignment\outputs\point4_minhash_pairs_ge_0_5_t{t}.csv
  Run 5: FP=71, FN=2

Running MinHash for t = 100
  Run 1: FP=55, FN=4
  Run 2: FP=43, FN=1
  Run 3: FP=20, FN=3
  Run 4: FP=31, FN=3
Saved: C:\Nitin Awasthi\M-Tech\Semester-5\Big Data\CSL7110_Assignment\outputs\point4_minhash_pairs_ge_0_5_t{t}.csv
  Run 5: FP=34, FN=2

Running MinHash for t = 200
  Run 1: FP=11, FN=1
  Run 2: FP=14, FN=0
  Run 3: FP=1, FN=0
  Run 4: FP=9, FN=3
Saved: C:\Nitin Awasthi\M-Tech\Semester-5\Big Data\CSL7110_Assignment\outputs\point4_minhash_pairs_ge_0_5_t{t}.csv
  Run 5: FP=8, FN=3

Point 4 Summary:
     t  Avg_FP_5runs  Avg_FN_5runs                 FP_runs          FN_runs
0

5. LSH on MovieLens dataset [20]

Step 5.1 — LSH candidate generation from signatures

In [59]:
def lsh_candidates_from_signatures(sigs, rows_per_band, num_bands):
    """
    sigs: dict[user_id] -> signature np.array length t
    Candidate pair if two users collide in at least one band.
    """
    # Check signature length
    any_user = next(iter(sigs))
    t = len(sigs[any_user])
    assert rows_per_band * num_bands == t, "rows_per_band * num_bands must equal signature length"

    # Each band has hash buckets
    band_buckets = [defaultdict(list) for _ in range(num_bands)]

    for user_id, sig in sigs.items():
        for band_idx in range(num_bands):
            start = band_idx * rows_per_band
            end = start + rows_per_band
            band_tuple = tuple(sig[start:end].tolist())
            band_buckets[band_idx][band_tuple].append(user_id)

    # Generate candidate pairs from shared buckets
    candidates = set()
    for band_idx in range(num_bands):
        for bucket_users in band_buckets[band_idx].values():
            if len(bucket_users) > 1:
                for u1, u2 in combinations(sorted(bucket_users), 2):
                    candidates.add((u1, u2))

    return candidates

FP/FN helper for candidate pairs

In [61]:
def fp_fn_counts_from_candidate_pairs(exact_dict, candidate_pairs, threshold):
    """
    exact_dict: {(u1,u2): exact_jaccard}
    candidate_pairs: set of (u1,u2)
    """
    exact_pos = {pair for pair, s in exact_dict.items() if s >= threshold}
    pred_pos = set(candidate_pairs)

    fp = len(pred_pos - exact_pos)
    fn = len(exact_pos - pred_pos)
    return fp, fn, exact_pos, pred_pos

Run Point 5 (all configs, thresholds 0.6 and 0.8, average over 5 runs)

In [64]:

lsh_configs = [
    (50, 5, 10),
    (100, 5, 20),
    (200, 5, 40),
    (200, 10, 20),
]

thresholds = [0.6, 0.8]

summary_rows = []

for t, rows_per_band, num_bands in lsh_configs:
    print(f"\nLSH Config: t={t}, rows_per_band={rows_per_band}, num_bands={num_bands}")

    for thr in thresholds:
        fp_runs = []
        fn_runs = []

        for run_idx, seed in enumerate([201, 202, 203, 204, 205], start=1):
            # Build signatures (same way as Point 4, but now used for LSH)
            value_table = build_minhash_value_table_for_movies(max_movie_id, t=t, seed=seed, m=50021)
            sigs = user_signatures_from_movie_value_table(user_movies, users, value_table)

            # Get candidate pairs via LSH
            candidates = lsh_candidates_from_signatures(sigs, rows_per_band, num_bands)

            # Compare candidate pairs vs exact positives at threshold thr
            fp, fn, exact_pos, pred_pos = fp_fn_counts_from_candidate_pairs(exact_ml, candidates, threshold=thr)

            fp_runs.append(fp)
            fn_runs.append(fn)

            print(f"  threshold={thr}, run={run_idx}: candidates={len(candidates)}, FP={fp}, FN={fn}")

        summary_rows.append({
            "t": t,
            "rows_per_band_r": rows_per_band,
            "num_bands_b": num_bands,
            "threshold": thr,
            "Avg_FP_5runs": float(np.mean(fp_runs)),
            "Avg_FN_5runs": float(np.mean(fn_runs)),
            "FP_runs": str(fp_runs),
            "FN_runs": str(fn_runs)
        })

df_point5_summary = pd.DataFrame(summary_rows)
print("\nPoint 5 Summary:")
print(df_point5_summary)

df_point5_summary.to_csv(r"C:\Nitin Awasthi\M-Tech\Semester-5\Big Data\CSL7110_Assignment\outputs\point5_lsh_fp_fn_summary.csv", index=False)
print("Saved: outputs/point5_lsh_fp_fn_summary.csv")


LSH Config: t=50, rows_per_band=5, num_bands=10
  threshold=0.6, run=1: candidates=499, FP=496, FN=0
  threshold=0.6, run=2: candidates=542, FP=539, FN=0
  threshold=0.6, run=3: candidates=702, FP=699, FN=0
  threshold=0.6, run=4: candidates=532, FP=530, FN=1
  threshold=0.6, run=5: candidates=330, FP=327, FN=0
  threshold=0.8, run=1: candidates=499, FP=498, FN=0
  threshold=0.8, run=2: candidates=542, FP=541, FN=0
  threshold=0.8, run=3: candidates=702, FP=701, FN=0
  threshold=0.8, run=4: candidates=532, FP=531, FN=0
  threshold=0.8, run=5: candidates=330, FP=329, FN=0

LSH Config: t=100, rows_per_band=5, num_bands=20
  threshold=0.6, run=1: candidates=1298, FP=1295, FN=0
  threshold=0.6, run=2: candidates=929, FP=926, FN=0
  threshold=0.6, run=3: candidates=840, FP=837, FN=0
  threshold=0.6, run=4: candidates=1161, FP=1158, FN=0
  threshold=0.6, run=5: candidates=1801, FP=1799, FN=1
  threshold=0.8, run=1: candidates=1298, FP=1297, FN=0
  threshold=0.8, run=2: candidates=929, FP=92